# Per Capita Intensity Analysis

## Import Data And Libraries

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import os
from pathlib import Path
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')

from sqlalchemy import create_engine, text

In [4]:
env_path =Path.cwd()/"github_repo_cloned"/"india-logistics-hub-optimizer"/".env"
load_dotenv(env_path) 

HOST = os.getenv("DB_HOST")
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
OUTPUT_DIR = os.getenv("OUTPUT_DIR")

In [5]:
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}")


def sql(query, engine = engine):
    with engine.connect() as conn:
        df = pd.read_sql(text(query),conn)
    return df

In [9]:
query = 'select * from pca_data'

bill = sql(query)


query = 'select * from population_df'
population = sql(query)

In [10]:
bill.head()

,state,year,volume,amount
0,jammu and kashmir,2019,3819444.0,104713.89
1,himachal pradesh,2019,9104595.0,267974.40
2,punjab,2019,33637888.0,883716.86
3,chandigarh,2019,2487785.0,46901.04
4,uttarakhand,2019,14770838.0,438641.00


In [11]:
population.head()

,state,year,population
0,andaman & nicobar islands,2019,1504000
1,andaman & nicobar islands,2020,1537000
2,andaman & nicobar islands,2021,1571000
3,andaman & nicobar islands,2022,1608000
4,andaman & nicobar islands,2023,1646000


## Initial Data Checking

In [12]:
bill.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   state   265 non-null    object 
 1   year    265 non-null    int64  
 2   volume  265 non-null    float64
 3   amount  265 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 8.4+ KB


In [13]:
population.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   state       265 non-null    object
 1   year        265 non-null    int64 
 2   population  265 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 6.3+ KB


In [14]:
bill['state'].unique()

array(['jammu and kashmir', 'himachal pradesh', 'punjab', 'chandigarh',
       'uttarakhand', 'haryana', 'delhi', 'rajasthan', 'uttar pradesh',
       'bihar', 'sikkim', 'arunachal pradesh', 'nagaland', 'manipur',
       'mizoram', 'tripura', 'meghalaya', 'assam', 'west bengal',
       'jharkhand', 'odisha', 'chhattisgarh', 'madhya pradesh', 'gujarat',
       'daman and diu', 'dadra and nagar haveli', 'maharashtra',
       'karnataka', 'goa', 'lakshadweep', 'kerala', 'tamil nadu',
       'puducherry', 'andaman and nicobar', 'telangana', 'andhra pradesh',
       'other territory', 'ladakh'], dtype=object)

In [15]:
population['state'].unique()

array(['andaman & nicobar islands', 'andhra pradesh', 'arunachal pradesh',
       'assam', 'bihar', 'chandigarh', 'chhattisgarh',
       'dadra & nagar haveli', 'daman & diu', 'goa', 'gujarat', 'haryana',
       'himachal pradesh', 'india', 'jammu & kashmir', 'jharkhand',
       'karnataka', 'kerala', 'ladakh', 'lakshadweep', 'madhya pradesh',
       'maharashtra', 'manipur', 'meghalaya', 'mizoram', 'nagaland',
       'nct of delhi', 'odisha', 'puducherry', 'punjab', 'rajasthan',
       'sikkim', 'tamil nadu', 'telangana', 'tripura', 'uttar pradesh',
       'uttarakhand', 'west bengal'], dtype=object)

In [23]:
print("Difference In Bill:")
list(set(bill['state'])-set(population['state']))

Difference In Bill:


['daman and diu',
 'other territory',
 'dadra and nagar haveli',
 'andaman and nicobar',
 'jammu and kashmir',
 'delhi']

In [24]:
print("Difference In population:")
list(set(population['state'])-set(bill['state']))

Difference In population:


['andaman & nicobar islands',
 'daman & diu',
 'dadra & nagar haveli',
 'india',
 'jammu & kashmir',
 'nct of delhi']

In [45]:
bill_replace = {
    'daman and diu':'daman & diu',
    'dadra and nagar haveli':'dadra & nagar haveli',
    'jammu and kashmir': 'jammu & kashmir',
    'andaman and nicobar':'andaman & nicobar'
}

bill['state']=bill['state'].replace(bill_replace)
    

In [46]:
bill = bill[bill['state']!= 'other territory']

In [47]:
population_replace = {
    'nct of delhi':'delhi',
    'andaman & nicobar islands': 'andaman & nicobar'
}
population['state']=population['state'].replace(population_replace)

In [48]:
population = population[population['state'] != 'india']

In [49]:
print("Difference In Bill:")
list(set(bill['state'])-set(population['state']))

Difference In Bill:


[]

In [50]:
print("Difference In population:")
list(set(population['state'])-set(bill['state']))

Difference In population:


[]

In [52]:
bill.describe()

,year,volume,amount
count,258.000000,2.580000e+02,2.580000e+02
mean,2022.011628,3.461840e+07,9.874615e+05
std,1.998993,4.996572e+07,1.396332e+06
min,2019.000000,3.780000e+02,1.700000e+01
25%,2020.000000,9.749840e+05,3.179000e+04
50%,2022.000000,1.283892e+07,4.243522e+05
75%,2024.000000,4.611083e+07,1.423620e+06
max,2025.000000,2.813539e+08,7.929648e+06


In [53]:
population.describe()

,year,population
count,259.000000,2.590000e+02
mean,2022.000000,3.710779e+07
std,2.003872,4.868195e+07
min,2019.000000,6.800000e+04
25%,2020.000000,1.560500e+06
50%,2022.000000,2.096500e+07
75%,2024.000000,5.324800e+07
max,2025.000000,2.404680e+08


In [56]:
bill.shape

(258, 4)

In [57]:
population.shape

(259, 3)

In [61]:
bill['state'].value_counts().reset_index().query("count<7")

,state,count
36,ladakh,6


In [76]:
bill[bill['state']=='ladakh']

,state,year,volume,amount
73,ladakh,2020,55545.0,1684.64
111,ladakh,2021,100472.0,3042.28
149,ladakh,2022,159151.0,4724.08
187,ladakh,2023,197990.0,5673.14
225,ladakh,2024,223938.0,6354.70
249,ladakh,2025,212245.0,6299.87


In [63]:
population[population['state']=='ladakh']

,state,year,population
125,ladakh,2019,293000
126,ladakh,2020,295000
127,ladakh,2021,297000
128,ladakh,2022,299000
129,ladakh,2023,300000
130,ladakh,2024,302000
131,ladakh,2025,304000


In [70]:
mask = (population['state']== 'ladakh') & (population['year']==2019)


population = population[~mask]

In [71]:
population[population['state']=='ladakh']

,state,year,population
126,ladakh,2020,295000
127,ladakh,2021,297000
128,ladakh,2022,299000
129,ladakh,2023,300000
130,ladakh,2024,302000
131,ladakh,2025,304000


In [79]:
bill.groupby('state')['year'].count().reset_index().query('year !=7')

,state,year
18,ladakh,6


In [80]:
population.groupby('state')['year'].count().reset_index().query('year !=7')

,state,year
18,ladakh,6


In [81]:
bill.isnull().sum()

state     0
year      0
volume    0
amount    0
dtype: int64

In [82]:
population.isnull().sum()

state         0
year          0
population    0
dtype: int64

In [54]:
bill.head()

,state,year,volume,amount
0,jammu & kashmir,2019,3819444.0,104713.89
1,himachal pradesh,2019,9104595.0,267974.40
2,punjab,2019,33637888.0,883716.86
3,chandigarh,2019,2487785.0,46901.04
4,uttarakhand,2019,14770838.0,438641.00


In [55]:
population.head()

,state,year,population
0,andaman & nicobar,2019,1504000
1,andaman & nicobar,2020,1537000
2,andaman & nicobar,2021,1571000
3,andaman & nicobar,2022,1608000
4,andaman & nicobar,2023,1646000


## Analysis